# Deep Lagrangian Network — Spring Pendulum (2-DOF)

This notebook applies **Deep Lagrangian Networks (DeLaN)** to a 2-DOF nonlinear
system and compares it to a plain VanillaNN.

DeLaN is a structurally constrained LNN that decomposes the Lagrangian explicitly:

$$L(q, \dot{q}) = T(q, \dot{q}) - V(q) = \tfrac{1}{2}\dot{q}^\top M(q)\dot{q} - V(q)$$

Two architectural constraints are enforced **by construction**:
1. $M(q) = L_d(q)\,L_d(q)^\top + \varepsilon I$ — **symmetric positive-definite**
   via Cholesky factorisation (invertible mass matrix, physically meaningful dynamics)
2. $V(q) = \text{softplus}(V_{\text{net}}(q)) \geq 0$ — **non-negative potential energy**

These are not loss penalties — wrong outputs are architecturally unrepresentable.

**Key insight:** on a 2-DOF coupled system, the vanilla LNN's implicit mass matrix can
become near-singular or indefinite during rollout, destabilising the integration.
DeLaN's guaranteed positive-definiteness prevents this. Both outperform VanillaNN
in energy conservation; DeLaN is more robust.


In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
from systems import SpringPendulum
from models import DeLaN, VanillaNN
from utils.training import train_dynamics_model

SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE}")

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
COLORS = {"True": "#2c3e50", "DeLaN": "#2ecc71", "VanillaNN": "#e74c3c"}

## System: Spring Pendulum (2-DOF)

The spring pendulum is described in polar coordinates $q = (r, \theta)$:

$$\ddot{r} = r\dot{\theta}^2 - g(1-\cos\theta) - 2k(r-r_0)$$
$$\ddot{\theta} = \frac{-g\sin\theta - 2\dot{r}\dot{\theta}}{r}$$

with $g=10$, $k=10$, $r_0=1$.

This system couples radial oscillation and angular swing through the inertial terms
$r\dot{\theta}^2$ and $\dot{r}\dot{\theta}$. The coupling makes the mass matrix
position-dependent and non-trivial — a 2×2 function of $q$ that the VanillaNN must
learn implicitly with no structural guidance.

Training data: **80 samples** of $(q, \dot{q}, \ddot{q})$ from $t \in [0, 3\,\text{s}]$.
Evaluation: RK4 rollout to $t = 8\,\text{s}$ in Cartesian $(x, y)$ coordinates.


In [ ]:
sp    = SpringPendulum(g=10.0, k=10.0, r0=1.0)
Q0    = np.array([1.1, 0.5])
QDOT0 = np.array([0.0, 0.0])
T_TRAIN, T_EVAL = (0.0, 3.0), (0.0, 8.0)
N_TRAIN, N_EVAL = 80, 800

train_ds  = sp.generate_dataset(Q0, QDOT0, T_TRAIN, N_TRAIN)
test_traj = sp.rollout(Q0, QDOT0, T_EVAL, N_EVAL)

t_eval  = test_traj["t"]
q_true  = test_traj["q"]
qd_true = test_traj["qdot"]
xy_true = test_traj["xy"]
E_true  = sp.total_energy(q_true, qd_true)

# Visualise ground truth in Cartesian coordinates
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(t_eval, xy_true[:, 0], color=COLORS["True"], lw=1.5, label="x(t)")
ax1.plot(t_eval, xy_true[:, 1], color="steelblue", lw=1.5, ls="--", label="y(t)")
ax1.axvline(T_TRAIN[1], color="gray", ls=":", lw=1.2, label="train end")
ax1.set_xlabel("t (s)"); ax1.set_ylabel("position (m)"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.set_title("Ground truth trajectory")
ax2.plot(xy_true[:, 0], xy_true[:, 1], color=COLORS["True"], lw=1)
ax2.set_xlabel("x (m)"); ax2.set_ylabel("y (m)"); ax2.set_aspect("equal"); ax2.grid(True, alpha=0.3)
ax2.set_title("Orbital view")
plt.tight_layout(); plt.show()

## DeLaN vs VanillaNN

Both models use **3 hidden layers of 64 units** and are trained for 2500 epochs on
the same MSE acceleration loss.

**DeLaN** uses `activation=nn.Softplus` throughout — smooth activations allow the
second derivatives required by the Euler-Lagrange equations to be well-behaved.
The Cholesky construction ensures the mass matrix $M(q)$ is SPD regardless of the
network weights, so the linear system $M(q)\ddot{q} = \ldots$ is always solvable.

**VanillaNN** maps $(q, \dot{q}) \mapsto \ddot{q}$ directly — no structure imposed.


In [ ]:
EPOCHS = 2_500

models = {
    "DeLaN":     DeLaN(    n_dof=2, hidden_dim=64, n_layers=3, activation=nn.Softplus),
    "VanillaNN": VanillaNN(n_dof=2, hidden_dim=64, n_layers=3),
}
for name, model in models.items():
    print(f"\nTraining {name} ...")
    hist = train_dynamics_model(model, train_ds, n_epochs=EPOCHS,
                                batch_size=32, device=DEVICE, verbose=True, log_every=500)
    print(f"  final loss={hist['train_loss'][-1]:.4e}  t={hist['wall_time']:.1f}s")

In [ ]:
def rk4(model, q0, qdot0, t_array, device="cpu"):
    q, qdot = q0.copy(), qdot0.copy()
    qs = [q.copy()]
    def acc(qi, qdi):
        qt  = torch.tensor(qi[None],  dtype=torch.float32, device=device)
        qdt = torch.tensor(qdi[None], dtype=torch.float32, device=device)
        with torch.enable_grad():
            return model.predict_acceleration(qt, qdt).detach().cpu().numpy()[0]
    for dt in np.diff(t_array):
        k1v = qdot;              k1a = acc(q,              qdot)
        k2v = qdot + .5*dt*k1a; k2a = acc(q + .5*dt*k1v, qdot + .5*dt*k1a)
        k3v = qdot + .5*dt*k2a; k3a = acc(q + .5*dt*k2v, qdot + .5*dt*k2a)
        k4v = qdot +    dt*k3a; k4a = acc(q +    dt*k3v, qdot +    dt*k3a)
        q    = q    + (dt/6)*(k1v + 2*k2v + 2*k3v + k4v)
        qdot = qdot + (dt/6)*(k1a + 2*k2a + 2*k3a + k4a)
        qs.append(q.copy())
    return np.array(qs)

split = T_TRAIN[1]
in_m, out_m = t_eval <= split, t_eval > split

preds = {}
for name, model in models.items():
    model.eval()
    q_pred  = np.clip(rk4(model, Q0, QDOT0, t_eval, DEVICE), -20, 20)
    qd_pred = np.gradient(q_pred, t_eval, axis=0)
    xy_pred = SpringPendulum.polar_to_xy(q_pred)
    E_pred  = sp.total_energy(q_pred, qd_pred)
    rmse_in  = float(np.sqrt(np.mean((xy_pred[in_m]  - xy_true[in_m])**2)))
    rmse_out = float(np.sqrt(np.mean((xy_pred[out_m] - xy_true[out_m])**2)))
    e_drift  = float(np.mean(np.abs(E_pred - E_true[0]) / (np.abs(E_true[0]) + 1e-8)))
    preds[name] = {"xy": xy_pred, "E": E_pred, "interp": rmse_in, "extrap": rmse_out, "edrift": e_drift}
    print(f"{name:<12}  interp={rmse_in:.4f}  extrap={rmse_out:.4f}  E-drift={e_drift:.4f}")

gain = preds["VanillaNN"]["extrap"] / (preds["DeLaN"]["extrap"] + 1e-12)
print(f"\nDeLaN extrap improvement over VanillaNN: {gain:.1f}×")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
fig.suptitle(f"DeLaN vs VanillaNN — Spring Pendulum 2-DOF\n"
             f"{N_TRAIN} training pts from [0, 3 s]  |  shaded = extrapolation",
             fontsize=12, fontweight="bold")

for ax, ci, lbl in zip(axes[:2], [0, 1], ["x (m)", "y (m)"]):
    ax.plot(t_eval, xy_true[:, ci], color=COLORS["True"], lw=1, ls="--",
            label="Ground truth", zorder=5)
    for name in ["DeLaN", "VanillaNN"]:
        p = preds[name]
        ax.plot(t_eval, np.clip(p["xy"][:, ci], -5, 5), color=COLORS[name], lw=1.8,
                label=f"{name}  (extrap={p['extrap']:.4f})")
    ax.axvline(split, color="gray", ls=":", lw=1.4, label="train end")
    ax.axvspan(split, t_eval[-1], alpha=0.07, color="gray")
    ax.set_ylabel(lbl, fontsize=11); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[2]
for name in ["DeLaN", "VanillaNN"]:
    err = np.sqrt(np.sum((np.clip(preds[name]["xy"], -5, 5) - xy_true)**2, axis=1))
    ax.semilogy(t_eval, err + 1e-6, color=COLORS[name], lw=1.6, label=name)
ax.axvline(split, color="gray", ls=":", lw=1.4)
ax.axvspan(split, t_eval[-1], alpha=0.07, color="gray")
ax.set_ylabel("‖xy error‖  (log)", fontsize=11); ax.set_xlabel("Time (s)", fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "delan_vs_vanilla.png"), dpi=150, bbox_inches="tight")
plt.show()

## Energy Conservation

The decisive diagnostic for these conservative systems is **total energy** $E = T + V$.

The VanillaNN's predicted accelerations are unconstrained — nothing prevents them from
injecting or draining energy. DeLaN's Lagrangian structure makes energy-violating
trajectories architecturally impossible: the total energy drifts near zero throughout
the rollout.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axhline(E_true[0], color=COLORS["True"], lw=1.2, ls="--", label=f"True E = {E_true[0]:.3f}")
for name in ["DeLaN", "VanillaNN"]:
    ax.plot(t_eval, preds[name]["E"], color=COLORS[name], lw=1.6,
            label=f"{name}  (E-drift={preds[name]['edrift']:.4f})")
ax.axvline(split, color="gray", ls=":", lw=1.4, label="train end")
ax.axvspan(split, t_eval[-1], alpha=0.07, color="gray")
ax.set_xlabel("t (s)"); ax.set_ylabel("Total energy E = T + V", fontsize=11)
ax.set_title("Energy conservation during rollout", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "delan_vs_vanilla_energy.png"), dpi=150, bbox_inches="tight")
plt.show()